# Clipt v7 — Football Jersey OCR Training Pipeline

**PURPOSE:** Train 3 football-specific models to fix the #1 pipeline gap: football jersey number detection returns 0 detections on ALL football videos.

| Section | Model | Type | Output |
|---------|-------|------|--------|
| 1 | Football Jersey OCR v7 | YOLO detect | `football_jersey_ocr_v7.pt` |
| 2 | Navy Jersey Specialist v7 | YOLO detect | `navy_jersey_specialist_v7.pt` |
| 3 | Football Player Crop v7 | YOLO detect | `football_player_crop_v7.pt` |

**After training:** copy all files from Google Drive —> `reelapp/playerJerseyIdentification-master/app/model/`

**Estimated time:** ~8—12 hours on A100, ~15—24 hours on T4
**GPU required:** A100 recommended (T4 works but slower, imgsz=1280 needs batch=2—4)

### Why Football OCR Fails Today
1. **Dark jerseys** (navy/black) create near-zero contrast between number and fabric
2. **Helmets + shoulder pads** obstruct upper portion of chest numbers
3. **Camera distance** — 20—50 yards away, players are 30—150px crops at 720p
4. **Outdoor lighting** — shadows, glare, variable sun vs stadium lights
5. **Multi-color fonts** — outlines, drop shadows, 3D effects confuse digit boundary detection
6. **22 players** in tight formations at line of scrimmage

### Verified Roboflow Datasets
- `footballplayertracking/jerseynumberdetectordigitdetector` — 13,815 images, digit localization
- `dark-blue-jt0mg/jerseynumbers` — 826 images, 10 digit classes (0—9)
- `volleyai-actions/jersey-number-detection-s01j4` — 6,932 images, jersey numbers
- `augmented-startups/football-player-detection-kucab` — 1,232 images, football players
- `football-tracking/football-presnap-tracker` — football player detection
- `bronkscottema/football-players-zm06l` — football players

### Research References
- Koshkina & Elder (CVPR 2024): Scene text recognition approach, 91.4% accuracy
- Amazon/Seahawks (Frontiers AI 2022): Synthetic data + curriculum learning for football
- Grad et al. (CVPR 2025): Uncertainty-aware jersey number recognition
- Key insight: **Digit-wise (0—9) outperforms whole-number (0—99)** — generalizes to unseen combinations


---
## Section 0: Setup


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 0A — Install Packages
# ═══════════════════════════════════════════════════════════════
!pip install roboflow ultralytics pyyaml -q
print("\n" + "=" * 50)
print("All packages installed")
print("=" * 50)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 0B — Imports + Config
# ═══════════════════════════════════════════════════════════════
import os, sys, torch, shutil, glob, yaml, time, json, cv2, gc, traceback
import numpy as np
from pathlib import Path
from google.colab import userdata

# -- Directories --
DRIVE_SAVE_DIR = '/content/drive/MyDrive/clipt_v7_models'
DRIVE_CHECKPOINTS = '/content/drive/MyDrive/clipt_v7_models/checkpoints'
DATA_BASE = '/content/data'

# -- API Keys --
ROBOFLOW_API_KEY = ''
try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

# -- GPU Check --
assert torch.cuda.is_available(), 'NO GPU -- Runtime > Change runtime type > A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
HAS_A100 = vram_gb > 40
DEFAULT_BATCH = 16 if HAS_A100 else 8

os.makedirs(DATA_BASE, exist_ok=True)

print(f'GPU: {gpu_name} ({vram_gb:.1f}GB)')
print(f'Batch size: {DEFAULT_BATCH}')
print(f'Roboflow API key: {"loaded" if ROBOFLOW_API_KEY else "MISSING -- set in Colab Secrets"}')

TRAINING_LOG = []

# -- Roboflow download helper --
def safe_download(ws, proj, ver, name, fmt='yolov8'):
    """Download dataset with version fallback."""
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for v in [ver] + [i for i in range(1, 10) if i != ver]:
        try:
            ds = rf.workspace(ws).project(proj).version(v).download(fmt)
            count = len(glob.glob(f'{ds.location}/train/images/*'))
            print(f'  {name} v{v}: {count} images')
            return ds
        except Exception as e:
            print(f'  v{v} failed: {e}')
    print(f'  {name}: all versions failed')
    return None

# -- Drive save helper --
def save_model_to_drive(model_name, task='detect'):
    """Copy best.pt to Google Drive save directory."""
    paths = sorted(glob.glob(f'runs/{task}/{model_name}*/weights/best.pt'))
    if not paths:
        alt = f'{DRIVE_CHECKPOINTS}/{model_name}/best.pt'
        if os.path.exists(alt): paths = [alt]
    assert paths and os.path.exists(paths[-1]), f'No trained model found for {model_name}'
    src = paths[-1]
    dst = f'{DRIVE_SAVE_DIR}/{model_name}.pt'
    shutil.copy2(src, dst)
    sz = os.path.getsize(dst) / 1024 / 1024
    print(f'{"=" * 60}')
    print(f'{model_name}.pt SAVED TO DRIVE ({sz:.1f}MB)')
    print(f'{"=" * 60}')
    return dst

def backup_checkpoints(model_name, task='detect'):
    """Backup best.pt and last.pt to Drive checkpoints dir."""
    ckpt_dir = f'{DRIVE_CHECKPOINTS}/{model_name}'
    os.makedirs(ckpt_dir, exist_ok=True)
    for wt in ['best.pt', 'last.pt']:
        paths = sorted(glob.glob(f'runs/{task}/{model_name}*/weights/{wt}'))
        if paths: shutil.copy2(paths[-1], f'{ckpt_dir}/{wt}')
    torch.cuda.empty_cache(); gc.collect()

# -- Drive backup callback --
def make_drive_backup_callback(model_name):
    """Returns callback that copies checkpoints to Drive every 5 epochs."""
    def backup(trainer):
        epoch = trainer.epoch
        if (epoch + 1) % 5 == 0 or epoch == 0:
            save_dir = trainer.save_dir
            dest_dir = f'{DRIVE_CHECKPOINTS}/{model_name}'
            os.makedirs(dest_dir, exist_ok=True)
            for fname in ['weights/best.pt', 'weights/last.pt']:
                src = os.path.join(str(save_dir), fname)
                if os.path.exists(src):
                    shutil.copy2(src, f'{dest_dir}/{os.path.basename(fname)}')
            print(f'  Epoch {epoch+1}: backed up to Drive')
    return backup

# -- Safe train with OOM recovery --
def safe_train(model_name, data_path, epochs, imgsz=832, batch=DEFAULT_BATCH, **kwargs):
    """YOLO training with Drive backup, auto-resume, OOM retry, and early stopping."""
    if data_path is None:
        msg = f'SKIPPED {model_name} -- dataset unavailable'
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    # Resolve data.yaml path
    if hasattr(data_path, 'location'):
        data_yaml = f'{data_path.location}/data.yaml'
    elif os.path.isdir(str(data_path)):
        data_yaml = f'{data_path}/data.yaml'
    else:
        data_yaml = str(data_path)

    if not os.path.exists(data_yaml):
        msg = f'SKIPPED {model_name} -- data.yaml not found at {data_yaml}'
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    # Check for Drive backup to resume from
    base_name = model_name.replace('.pt', '')
    drive_last = f'{DRIVE_CHECKPOINTS}/{base_name}/last.pt'
    resume_from = None
    if os.path.exists(drive_last):
        size_mb = os.path.getsize(drive_last) / 1024 / 1024
        if size_mb > 1:
            resume_from = drive_last
            print(f'RESUMING {model_name} from Drive backup ({size_mb:.1f}MB)')

    attempts = [
        (batch, 'full batch'),
        (max(batch // 2, 2), 'half batch (OOM retry)'),
    ]

    for attempt_batch, label in attempts:
        try:
            print(f'\n{"=" * 60}')
            print(f'TRAINING: {model_name} ({label})')
            print(f'  data={data_yaml}')
            print(f'  epochs={epochs}, imgsz={imgsz}, batch={attempt_batch}')
            print(f'  patience=25, save_period=5, amp=True, cache=True')
            if resume_from:
                print(f'  Resuming from: {resume_from}')
            for k, v in kwargs.items():
                print(f'  {k}={v}')
            print(f'{"=" * 60}\n')

            torch.cuda.empty_cache()
            gc.collect()

            if resume_from:
                model = YOLO(resume_from)
            else:
                model = YOLO('yolov8m.pt')

            model.add_callback('on_train_epoch_end', make_drive_backup_callback(base_name))
            start = time.time()

            if resume_from:
                model.train(resume=True)
            else:
                model.train(
                    data=data_yaml,
                    epochs=epochs,
                    imgsz=imgsz,
                    batch=attempt_batch,
                    name=base_name,
                    device=0,
                    patience=25,
                    save_period=5,
                    amp=True,
                    cache=True,
                    **kwargs
                )

            elapsed = time.time() - start
            mins = elapsed / 60

            backup_checkpoints(base_name, 'detect')

            msg = f'{model_name} complete in {mins:.1f} min'
            print(msg)
            TRAINING_LOG.append((model_name, 'TRAINED', mins, msg))
            return True

        except torch.cuda.OutOfMemoryError:
            print(f'OOM on {model_name} with batch={attempt_batch}')
            torch.cuda.empty_cache()
            gc.collect()
            if attempt_batch == attempts[-1][0]:
                msg = f'{model_name} -- OOM even at batch={attempt_batch}'
                print(msg)
                TRAINING_LOG.append((model_name, 'OOM_FAIL', 0, msg))
                return False
            print('Retrying with smaller batch...')
            resume_from = None

        except Exception as e:
            msg = f'{model_name} -- error: {type(e).__name__}: {e}'
            print(msg)
            traceback.print_exc()
            TRAINING_LOG.append((model_name, 'ERROR', 0, msg))
            torch.cuda.empty_cache()
            gc.collect()
            return False

    return False

print(f'\nAll helpers loaded. Ready to train.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 0C — Mount Google Drive
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
print(f'Drive mounted -- models save to {DRIVE_SAVE_DIR}')


**Expected output:**
```
Drive mounted -- models save to /content/drive/MyDrive/clipt_v7_models
```
---
## Section 1: Football Jersey OCR v7
YOLO detect model — reads digits 0—9 on football jerseys.

**Why this model is needed:** All 5 current OCR layers return 0 detections on football videos. Existing models trained primarily on basketball/soccer data.

**Dataset strategy:**
1. `dark-blue-jt0mg/jerseynumbers` (826 images, 10 digit classes 0—9) — PRIMARY, has digit labels
2. `volleyai-actions/jersey-number-detection-s01j4` (6,932 images) — large jersey number dataset
3. `footballplayertracking/jerseynumberdetectordigitdetector` (13,815 images) — football-specific digit localization
4. Merged + remapped to 10 classes (0—9)

**Augmentation:** hsv_v=0.7, hsv_s=0.7, erasing=0.4, mosaic=0.9, copy_paste=0.3, scale=0.5

**Training:** YOLOv8m, 150 epochs, imgsz=832, batch=8


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1A — Download football jersey OCR datasets
# ═══════════════════════════════════════════════════════════════
from roboflow import Roboflow

assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY in Colab Secrets'

print('=== Downloading Football Jersey OCR Datasets ===')

# Dataset 1: Dark Blue JerseyNumbers -- 826 images, 10 digit classes (0-9)
ds_digits = safe_download('dark-blue-jt0mg', 'jerseynumbers', 5, 'jersey_digits_0_9')

# Dataset 2: VolleyAi jersey-number-detection -- 6,932 images
ds_volley = safe_download('volleyai-actions', 'jersey-number-detection-s01j4', 2, 'volley_jersey_numbers')

# Dataset 3: Football digit detector -- 13,815 images, digit localization (nc=1)
ds_football = safe_download('footballplayertracking', 'jerseynumberdetectordigitdetector', 1, 'football_digit_locator')

# -- Merge all datasets into unified 10-class format --
print('\nMerging datasets...')
MERGED = f'{DATA_BASE}/football_ocr_merged'
for s in ['train', 'val']:
    os.makedirs(f'{MERGED}/{s}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{s}/labels', exist_ok=True)

DIGIT_NAMES = {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}
total_images = 0

def copy_dataset(ds, prefix, remap_to_class=None):
    """Copy images+labels from a dataset into the merged folder."""
    global total_images
    if not ds:
        print(f'  Skipping {prefix} -- download failed')
        return
    loc = ds.location
    for sp in ['train', 'valid', 'val', 'test']:
        idir = f'{loc}/{sp}/images'; ldir = f'{loc}/{sp}/labels'
        if not os.path.exists(idir): continue
        target = 'val' if sp in ['valid', 'val', 'test'] else 'train'
        for img_path in glob.glob(f'{idir}/*'):
            fn = f'{prefix}_{os.path.basename(img_path)}'
            shutil.copy2(img_path, f'{MERGED}/{target}/images/{fn}')
            lbl = os.path.join(ldir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
            if os.path.exists(lbl):
                with open(lbl) as f:
                    lines = f.readlines()
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        if remap_to_class is not None:
                            cls_id = remap_to_class
                        if 0 <= cls_id <= 9:
                            new_lines.append(f'{cls_id} {" ".join(parts[1:])}')
                        else:
                            new_lines.append(f'0 {" ".join(parts[1:])}')
                if new_lines:
                    with open(f'{MERGED}/{target}/labels/{os.path.splitext(fn)[0]}.txt', 'w') as f:
                        f.write('\n'.join(new_lines) + '\n')
            total_images += 1

copy_dataset(ds_digits, 'digits')       # Already has 0-9 classes
copy_dataset(ds_volley, 'volley')       # Jersey numbers
copy_dataset(ds_football, 'football', remap_to_class=0)  # nc=1, remap to class 0

# -- Write data.yaml --
dy = {
    'path': MERGED,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 10,
    'names': DIGIT_NAMES
}
with open(f'{MERGED}/data.yaml', 'w') as f:
    yaml.dump(dy, f)

tc = len(glob.glob(f'{MERGED}/train/images/*'))
vc = len(glob.glob(f'{MERGED}/val/images/*'))
print(f'\nFootball OCR dataset: {tc} train, {vc} val -- 10 classes (digits 0-9)')
print(f'Total images merged: {total_images}')
assert tc >= 100, f'Need 100+ training images, have {tc}'
print('Ready for training')


**Expected output:**
```
=== Downloading Football Jersey OCR Datasets ===
  jersey_digits_0_9 v5: ~826 images
  volley_jersey_numbers v2: ~6932 images
  football_digit_locator v1: ~13815 images

Merging datasets...
Football OCR dataset: ~18000 train, ~3500 val — 10 classes (digits 0-9)
Ready for training
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1B — Train football_jersey_ocr_v7
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO

MODEL_NAME = 'football_jersey_ocr_v7'
DATA_PATH = f'{DATA_BASE}/football_ocr_merged/data.yaml'

assert os.path.exists(DATA_PATH), 'No OCR data -- run Cell 1A first'
tc = len(glob.glob(f'{DATA_BASE}/football_ocr_merged/train/images/*'))
assert tc >= 100, f'Need 100+ images, have {tc}'
print(f'{tc} training images ready')

# Guard: check if already trained
drive_model = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'
if os.path.exists(drive_model):
    sz = os.path.getsize(drive_model) / 1024 / 1024
    print(f'Model already exists on Drive ({sz:.1f}MB) -- delete it to retrain')
else:
    result = safe_train(
        MODEL_NAME,
        DATA_PATH,
        epochs=150,
        imgsz=832,
        batch=DEFAULT_BATCH,
        # Football-specific augmentation
        hsv_v=0.7,        # Heavy brightness variation (dark jerseys + outdoor shadows)
        hsv_s=0.7,        # Heavy saturation variation (different team colors)
        erasing=0.4,       # Simulate helmet/pad obstruction
        mosaic=0.9,        # Strong mosaic for context variety
        copy_paste=0.3,    # Copy-paste augmentation
        scale=0.5,         # Scale variation (near/far camera)
        flipud=0.1,        # Slight vertical flip
    )
    if result:
        print(f'\n{MODEL_NAME} training PASSED')
    else:
        print(f'\n{MODEL_NAME} training FAILED -- check logs above')


**Expected output:**
```
18000 training images ready

============================================================
TRAINING: football_jersey_ocr_v7 (full batch)
  epochs=150, imgsz=832, batch=8-16
  patience=25, save_period=5, amp=True, cache=True
============================================================

... (150 epochs, ~4-6 hours on A100, ~6-9 hours on T4) ...

football_jersey_ocr_v7 training PASSED
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1C — Save football_jersey_ocr_v7.pt to Drive
# ═══════════════════════════════════════════════════════════════
save_model_to_drive('football_jersey_ocr_v7', 'detect')


**Expected output:**
```
============================================================
football_jersey_ocr_v7.pt SAVED TO DRIVE (~50MB)
============================================================
```
---
## Section 2: Navy Jersey Specialist v7
YOLO detect model — specifically trained for DARK jerseys (navy, black, dark green, dark red).

**Why this model is needed:** Dark jerseys are the #1 failure mode. Navy/black fabric creates near-zero contrast. The existing `dark_jersey_specialist_v3` was not football-specific.

**Dataset strategy:**
1. Reuse merged dataset from Section 1
2. Generate dark-augmented copies (brightness reduction + CLAHE + darken)
3. ~3x the original dataset size with dark variants

**Augmentation (MAXIMUM dark emphasis):**
- `hsv_v=0.95` — extreme brightness variation
- `hsv_s=0.95` — extreme saturation variation
- `erasing=0.7` — heavy random erasing (severe obstruction)
- `copy_paste=0.6` — heavy copy-paste for multi-player overlap
- `mosaic=1.0`, `scale=0.6`

**Training:** YOLOv8m, 150 epochs, imgsz=832, batch=8


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2A — Prepare navy jersey specialist dataset
# ═══════════════════════════════════════════════════════════════
print('=== Preparing Navy Jersey Specialist Dataset ===')

# Reuse datasets from Section 1 if available
src_base = f'{DATA_BASE}/football_ocr_merged'
if not os.path.exists(f'{src_base}/data.yaml'):
    print('Section 1 data not found -- downloading fresh...')
    ds_digits_2 = safe_download('dark-blue-jt0mg', 'jerseynumbers', 5, 'navy_digits')
    ds_football_2 = safe_download('footballplayertracking', 'jerseynumberdetectordigitdetector', 1, 'navy_football')
    src_base = None
    print('WARNING: Run Section 1 first for best results')
else:
    print('Reusing merged dataset from Section 1')

# -- Generate dark-augmented copies --
print('\nGenerating dark-augmented training images...')
NAVY_DIR = f'{DATA_BASE}/navy_specialist'
for s in ['train', 'val']:
    os.makedirs(f'{NAVY_DIR}/{s}/images', exist_ok=True)
    os.makedirs(f'{NAVY_DIR}/{s}/labels', exist_ok=True)

if src_base and os.path.exists(src_base):
    dark_count = 0
    for sp in ['train', 'val']:
        src_imgs = glob.glob(f'{src_base}/{sp}/images/*')
        for img_path in src_imgs:
            fn = os.path.basename(img_path)
            lbl_src = f'{src_base}/{sp}/labels/{os.path.splitext(fn)[0]}.txt'

            # Copy original
            shutil.copy2(img_path, f'{NAVY_DIR}/{sp}/images/orig_{fn}')
            if os.path.exists(lbl_src):
                shutil.copy2(lbl_src, f'{NAVY_DIR}/{sp}/labels/orig_{os.path.splitext(fn)[0]}.txt')

            # Generate dark variants
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue

                # Dark variant 1: heavy brightness reduction (30-60%)
                dark_factor = np.random.uniform(0.3, 0.6)
                dark = (img.astype(np.float32) * dark_factor).clip(0, 255).astype(np.uint8)
                cv2.imwrite(f'{NAVY_DIR}/{sp}/images/dark_{fn}', dark)
                if os.path.exists(lbl_src):
                    shutil.copy2(lbl_src, f'{NAVY_DIR}/{sp}/labels/dark_{os.path.splitext(fn)[0]}.txt')
                dark_count += 1

                # Dark variant 2: CLAHE then darken (simulates stadium lighting)
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                clahe = cv2.createCLAHE(clipLimit=40, tileGridSize=(8, 8))
                enhanced = clahe.apply(gray)
                enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
                dark2 = (enhanced_bgr.astype(np.float32) * np.random.uniform(0.4, 0.7)).clip(0, 255).astype(np.uint8)
                cv2.imwrite(f'{NAVY_DIR}/{sp}/images/clahe_{fn}', dark2)
                if os.path.exists(lbl_src):
                    shutil.copy2(lbl_src, f'{NAVY_DIR}/{sp}/labels/clahe_{os.path.splitext(fn)[0]}.txt')
                dark_count += 1
            except Exception:
                pass

    print(f'Generated {dark_count} dark-augmented images')

# -- Write data.yaml --
tc_navy = len(glob.glob(f'{NAVY_DIR}/train/images/*'))
vc_navy = len(glob.glob(f'{NAVY_DIR}/val/images/*'))

dy = {
    'path': NAVY_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 10,
    'names': {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}
}
with open(f'{NAVY_DIR}/data.yaml', 'w') as f:
    yaml.dump(dy, f)

print(f'\nNavy specialist dataset: {tc_navy} train, {vc_navy} val')
assert tc_navy >= 100, f'Need 100+ training images, have {tc_navy}'
print('Ready for training')


**Expected output:**
```
=== Preparing Navy Jersey Specialist Dataset ===
Reusing merged dataset from Section 1
Generating dark-augmented training images...
Generated ~40000 dark-augmented images

Navy specialist dataset: ~54000 train, ~10500 val
Ready for training
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2B — Train navy_jersey_specialist_v7
# ═══════════════════════════════════════════════════════════════
MODEL_NAME = 'navy_jersey_specialist_v7'
DATA_PATH = f'{DATA_BASE}/navy_specialist/data.yaml'

assert os.path.exists(DATA_PATH), 'No navy data -- run Cell 2A first'
tc = len(glob.glob(f'{DATA_BASE}/navy_specialist/train/images/*'))
assert tc >= 100, f'Need 100+ images, have {tc}'
print(f'{tc} training images ready')

# Guard: check if already trained
drive_model = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'
if os.path.exists(drive_model):
    sz = os.path.getsize(drive_model) / 1024 / 1024
    print(f'Model already exists on Drive ({sz:.1f}MB) -- delete it to retrain')
else:
    result = safe_train(
        MODEL_NAME,
        DATA_PATH,
        epochs=150,
        imgsz=832,
        batch=DEFAULT_BATCH,
        # MAXIMUM dark augmentation
        hsv_v=0.95,        # Extreme brightness variation
        hsv_s=0.95,        # Extreme saturation variation
        erasing=0.7,       # Heavy random erasing (severe obstruction)
        copy_paste=0.6,    # Heavy copy-paste (multi-player overlap)
        mosaic=1.0,        # Maximum mosaic
        scale=0.6,         # Wide scale variation
        flipud=0.1,
    )
    if result:
        print(f'\n{MODEL_NAME} training PASSED')
    else:
        print(f'\n{MODEL_NAME} training FAILED -- check logs above')


**Expected output:**
```
54000 training images ready

============================================================
TRAINING: navy_jersey_specialist_v7 (full batch)
  epochs=150, imgsz=832, batch=8-16
  hsv_v=0.95, hsv_s=0.95, erasing=0.7, copy_paste=0.6
============================================================

... (150 epochs, ~6-9 hours on A100 due to larger dataset) ...

navy_jersey_specialist_v7 training PASSED
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2C — Save navy_jersey_specialist_v7.pt to Drive
# ═══════════════════════════════════════════════════════════════
save_model_to_drive('navy_jersey_specialist_v7', 'detect')


**Expected output:**
```
============================================================
navy_jersey_specialist_v7.pt SAVED TO DRIVE (~50MB)
============================================================
```
---
## Section 3: Football Player Crop v7
YOLO detect model — detects individual football players at broadcast distances (30—150px at 720p).

**Why this model is needed:** The current `player_detector_v5` returns 501 detections for football frames, with 99.8% being tiny noise bounding boxes (2x5, 1x4 pixels). A football-specific player detector will produce clean bounding boxes at broadcast camera distances (20—50 yards).

**Dataset strategy:**
1. `augmented-startups/football-player-detection-kucab` (1,232 images) — football players
2. `football-tracking/football-presnap-tracker` — football player detection
3. `bronkscottema/football-players-zm06l` — football players
4. Merged into single `player` class

**Training:** YOLOv8m, 100 epochs, imgsz=1280, batch=4 (T4) or 8 (A100)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3A — Download football player detection datasets
# ═══════════════════════════════════════════════════════════════
print('=== Downloading Football Player Detection Datasets ===')

ds_kucab = safe_download('augmented-startups', 'football-player-detection-kucab', 1, 'football_players_kucab')
ds_presnap = safe_download('football-tracking', 'football-presnap-tracker', 1, 'football_presnap')
ds_bronks = safe_download('bronkscottema', 'football-players-zm06l', 15, 'football_players_bronks')

# -- Merge into single dataset with 1 class: player --
print('\nMerging football player datasets...')
PLAYER_DIR = f'{DATA_BASE}/football_player_merged'
for s in ['train', 'val']:
    os.makedirs(f'{PLAYER_DIR}/{s}/images', exist_ok=True)
    os.makedirs(f'{PLAYER_DIR}/{s}/labels', exist_ok=True)

total_player_images = 0

def copy_player_dataset(ds, prefix):
    """Copy images+labels, remap all classes to 0 (player)."""
    global total_player_images
    if not ds:
        print(f'  Skipping {prefix} -- download failed')
        return
    loc = ds.location
    for sp in ['train', 'valid', 'val', 'test']:
        idir = f'{loc}/{sp}/images'; ldir = f'{loc}/{sp}/labels'
        if not os.path.exists(idir): continue
        target = 'val' if sp in ['valid', 'val', 'test'] else 'train'
        for img_path in glob.glob(f'{idir}/*'):
            fn = f'{prefix}_{os.path.basename(img_path)}'
            shutil.copy2(img_path, f'{PLAYER_DIR}/{target}/images/{fn}')
            lbl = os.path.join(ldir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
            if os.path.exists(lbl):
                with open(lbl) as f:
                    lines = f.readlines()
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        new_lines.append(f'0 {" ".join(parts[1:])}')
                if new_lines:
                    with open(f'{PLAYER_DIR}/{target}/labels/{os.path.splitext(fn)[0]}.txt', 'w') as f:
                        f.write('\n'.join(new_lines) + '\n')
            total_player_images += 1

copy_player_dataset(ds_kucab, 'kucab')
copy_player_dataset(ds_presnap, 'presnap')
copy_player_dataset(ds_bronks, 'bronks')

# -- Write data.yaml --
dy = {
    'path': PLAYER_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': {0: 'player'}
}
with open(f'{PLAYER_DIR}/data.yaml', 'w') as f:
    yaml.dump(dy, f)

tc_p = len(glob.glob(f'{PLAYER_DIR}/train/images/*'))
vc_p = len(glob.glob(f'{PLAYER_DIR}/val/images/*'))
print(f'\nFootball player dataset: {tc_p} train, {vc_p} val -- 1 class (player)')
print(f'Total images merged: {total_player_images}')
assert tc_p >= 50, f'Need 50+ training images, have {tc_p}'
print('Ready for training')


**Expected output:**
```
=== Downloading Football Player Detection Datasets ===
  football_players_kucab v1: ~1232 images
  football_presnap v1: ~500 images
  football_players_bronks v15: ~800 images

Merging football player datasets...
Football player dataset: ~2000 train, ~500 val — 1 class (player)
Ready for training
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3B — Train football_player_crop_v7
# ═══════════════════════════════════════════════════════════════
MODEL_NAME = 'football_player_crop_v7'
DATA_PATH = f'{DATA_BASE}/football_player_merged/data.yaml'

assert os.path.exists(DATA_PATH), 'No player data -- run Cell 3A first'
tc = len(glob.glob(f'{DATA_BASE}/football_player_merged/train/images/*'))
assert tc >= 50, f'Need 50+ images, have {tc}'
print(f'{tc} training images ready')

# imgsz=1280 for small player detection at broadcast distances
# Batch = 4 on T4 (16GB), 8 on A100 (80GB)
player_batch = 8 if HAS_A100 else 4

# Guard: check if already trained
drive_model = f'{DRIVE_SAVE_DIR}/{MODEL_NAME}.pt'
if os.path.exists(drive_model):
    sz = os.path.getsize(drive_model) / 1024 / 1024
    print(f'Model already exists on Drive ({sz:.1f}MB) -- delete it to retrain')
else:
    result = safe_train(
        MODEL_NAME,
        DATA_PATH,
        epochs=100,
        imgsz=1280,          # Large image size for small players at distance
        batch=player_batch,
        # Standard augmentation for player detection
        mosaic=0.9,
        scale=0.5,
        hsv_v=0.4,
        hsv_s=0.4,
    )
    if result:
        print(f'\n{MODEL_NAME} training PASSED')
    else:
        print(f'\n{MODEL_NAME} training FAILED -- check logs above')


**Expected output:**
```
2000 training images ready

============================================================
TRAINING: football_player_crop_v7 (full batch)
  epochs=100, imgsz=1280, batch=4-8
============================================================

... (100 epochs, ~5-8 hours on A100, ~10-15 hours on T4) ...

football_player_crop_v7 training PASSED
```


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3C — Save football_player_crop_v7.pt to Drive
# ═══════════════════════════════════════════════════════════════
save_model_to_drive('football_player_crop_v7', 'detect')


**Expected output:**
```
============================================================
football_player_crop_v7.pt SAVED TO DRIVE (~50MB)
============================================================
```
---
## Section 4: Final Report
Verify all 3 models are saved to Google Drive and print copy instructions.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4A — Check Drive folder
# ═══════════════════════════════════════════════════════════════
print(f'\nChecking {DRIVE_SAVE_DIR}...\n')
all_files = sorted(glob.glob(f'{DRIVE_SAVE_DIR}/*.*'))
if all_files:
    total_size = 0
    for f in all_files:
        sz = os.path.getsize(f) / 1024 / 1024; total_size += sz
        print(f'  {os.path.basename(f):45s} {sz:8.1f}MB')
    print(f'\n  Total: {len(all_files)} files, {total_size:.1f}MB')
else:
    print('  No files found -- run Sections 1-3')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4B — Final Status Table
# ═══════════════════════════════════════════════════════════════
ALL_MODELS = [
    ('football_jersey_ocr_v7.pt', 'YOLO detect', 'Football jersey digit OCR (0-9)'),
    ('navy_jersey_specialist_v7.pt', 'YOLO detect', 'Dark jersey specialist (navy/black)'),
    ('football_player_crop_v7.pt', 'YOLO detect', 'Football player detection at broadcast distance'),
]

print('=' * 70)
print('  CLIPT V7 TRAINING COMPLETE -- FOOTBALL OCR FIX')
print('=' * 70)

print(f'\nGOOGLE DRIVE ({DRIVE_SAVE_DIR}):')
print(f'   Open Google Drive > MyDrive > clipt_v7_models')
print(f'   Copy ALL files to: reelapp/playerJerseyIdentification-master/app/model/')

saved = 0; missing = 0
print(f'\n{"Model":<45s} {"Type":<15s} {"Status":<10s} {"Size":<10s}')
print('-' * 75)

for mn, mtype, desc in ALL_MODELS:
    dp = f'{DRIVE_SAVE_DIR}/{mn}'
    if os.path.exists(dp):
        sz = os.path.getsize(dp) / 1024 / 1024
        print(f'{mn:<45s} {mtype:<15s} SAVED      {sz:>7.1f}MB')
        saved += 1
    else:
        print(f'{mn:<45s} {mtype:<15s} MISSING')
        missing += 1

print('-' * 75)
print(f'\n{saved}/{len(ALL_MODELS)} models saved')
if missing > 0:
    print(f'{missing} models still need training')
else:
    print(f'\nALL MODELS TRAINED!')

# Training log
if TRAINING_LOG:
    print(f'\nTraining Log:')
    for name, status, mins, msg in TRAINING_LOG:
        print(f'  {name:<40s} {status:<10s} {msg}')

print(f'\nNEXT STEPS:')
print(f'  1. Download all files from Google Drive > clipt_v7_models/')
print(f'  2. Copy to: reelapp/playerJerseyIdentification-master/app/model/')
print(f'  3. Wire models into roboflow_detector.py and analyze_pipeline.py')
print(f'  4. Deploy to Railway')
print(f'  5. Test with: curl POST /analyze football video')
print('=' * 70)


**Expected output:**
```
======================================================================
  CLIPT V7 TRAINING COMPLETE — FOOTBALL OCR FIX
======================================================================

Model                                         Type            Status     Size
---------------------------------------------------------------------------
football_jersey_ocr_v7.pt                     YOLO detect     SAVED       ~50MB
navy_jersey_specialist_v7.pt                  YOLO detect     SAVED       ~50MB
football_player_crop_v7.pt                    YOLO detect     SAVED       ~50MB
---------------------------------------------------------------------------

3/3 models saved

ALL MODELS TRAINED!
```
